# 06 — Fractal dimension & architectural self-similarity

Reproduces **SI Figure S8** of Eloy et al. *Nat Commun* **8**, 1014
(2017): the quantitative self-similarity figure for one of the evolved
champion genomes.

Five panels:

- **(a)** A mature tree (400 yr) coloured by Strahler order.
- **(b)** Architectural ratios across Horton stream orders: the
  bifurcation ratio $R_n$, the length ratio $R_l$, the diameter ratio
  $R_d$, the area ratio $R_a$, and the fractal dimension
  $D = \log R_n / \log R_l$. Open markers = young tree (25 yr),
  filled markers = mature tree (400 yr).
- **(c)** Time evolution of $R_n$, $R_l$, $D$ from one 500-generation
  simulation.
- **(d)** Branch tapering: per-segment scatter of length vs diameter
  with reference slopes 2/3, 1, 3/2.
- **(e)** Area conservation (Leonardo's rule) at every binary
  bifurcation, mean ≈ 0.95.

Companion of the qualitative diagnostics notebook
[05_strahler_diagnostics.ipynb](05_strahler_diagnostics.ipynb), which
already plots per-Strahler-order counts and the Leonardo histogram. The
new piece here is **per-Horton-stream** length, the four log-linear
fits, and the time evolution of the ratios.


In [ ]:
from dataclasses import replace
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

import mechatree as mt

mt.figstyle.apply()

CHAMPION_PATH = Path("../data/S3_champions.json").resolve()
SPECIES_ID = 0

cfg = mt.load_config(Path("../examples/forest.yaml").resolve())

# load_champion returns four pieces. The first three are the *coding* part of
# the genome (drive tree shape):
#   - safety    : 10-weight NN for the per-branch safety target
#   - allocation: 18-weight NN for (p_seeds, p_leaves, phototropism)
#   - angles    : {theta1, theta2, gamma1, gamma2} decoded from Fortran
#                 genome[0..2] (mod_tree.f90:108-111). Applying them to
#                 cfg.tree overrides the YAML defaults — without this the
#                 tree would grow with pi/4 / -pi/4 / 0 / pi.
# The fourth piece is *non-coding* bookkeeping (species_id, centroid_tag, ...).
safety, allocation, angles, non_coding = mt.load_champion(CHAMPION_PATH, species_id=SPECIES_ID)
cfg = replace(cfg, tree=replace(cfg.tree, **angles))

SEED = 42
GEN_YOUNG = 25
GEN_MATURE = 500
GEN_MAX = 1000
print(f"Champion species_id={non_coding['species_id']} ({non_coding.get('centroid_tag', '?')})")
print(
    f"  branching angles: theta1={np.degrees(angles['theta1']):.2f}°  "
    f"theta2={np.degrees(angles['theta2']):.2f}°  "
    f"gamma1={np.degrees(angles['gamma1']):.2f}°  "
    f"gamma2={np.degrees(angles['gamma2']):.2f}°"
)
print(f"  leaf_transparency (tau): {cfg.light.leaf_transparency}")

## Setup

We use champion **species 0** from
[`data/S3_champions.json`](../data/S3_champions.json) (swap to
`species_id=1` to compare the second cluster). Two independent
simulations with the same seed:

- **400 generations** → final tree for panel (a) and the *filled*
  markers in panel (b).
- **500 generations with `on_step`** → snapshot history for panel (c)
  and the *open* markers in panel (b) at the 25-yr early snapshot.

The two vertical guides in panel (c) sit at 25 yr and 400 yr — the same
two ages that anchor panel (b)'s open / filled marker pair.


In [ ]:
def snapshot_cb(every: int = 5):
    """Closure: returns an `on_step` callable + a growing history list.

    Every `every` generations we snapshot:

    - ``horton_summary``  — per-Horton-stream view (kept for diameter / area
      means used by R_d / R_a fits).
    - ``n_w``             — classical Horton-Strahler stream count per order
      (``horton_strahler_counts``; the Fortran's ``NsegmentsS``). The paper's
      "# branches of order w" trace. ``n_w[0] == leaves``.
    - ``L_w``             — mean stream length per Strahler rank in unit twigs
      (``mean_stream_length``; the Fortran's ``length_Strahler``). The paper's
      ``<l>`` trace and the right series for the ``R_l`` fit.

    Panel (c) replays the history to plot R_n(t), R_l(t), D(t).
    """
    history: list[dict] = []

    def cb(gen: int, tree) -> None:
        if gen % every == 0:
            history.append(
                {
                    "gen": gen,
                    "summary": mt.horton_summary(tree),
                    "n_w": mt.horton_strahler_counts(tree),
                    "L_w": mt.mean_stream_length(tree),
                }
            )

    return cb, history


def per_branch_arrays(tree):
    """Per-branch (strahler, <l>, diameter) arrays for panel (d).

    `<l>` is the recursive ``distance_to_leaves`` (Fortran
    ``b%distance_leaves`` in ``legacy_fortran/mod_tree.f90:1174-1203``),
    not the raw segment length — every MechaTree segment is a unit twig
    so segment length alone collapses panel (d) to y = 1.
    """
    tree.set_strahler()
    n = tree.get_number_of_branches()
    strahler = np.empty(n, dtype=np.int32)
    diameter = np.empty(n, dtype=np.float64)
    for i in range(n):
        strahler[i] = tree.get_strahler(i)
        diameter[i] = tree.get_diameter(i)
    length = mt.distance_to_leaves(tree)
    return strahler, length, diameter


def bifurcation_table(tree):
    """Panel (e): per-binary-junction (parent_<l>, area_ratio).

    `area_ratio = (d_left^2 + d_right^2) / d_parent^2` — the
    cross-section ratio that should be ≈ 1 if Leonardo's rule holds.
    The x-axis tag is the parent's recursive `<l>` (``distance_to_leaves``)
    rather than its raw segment length, mirroring SI Fig. S8(e)."""
    n = tree.get_number_of_branches()
    dist = mt.distance_to_leaves(tree)
    rows = []
    for i in range(n):
        kids = tree.get_children_index(i)
        if len(kids) != 2:
            continue
        d_p = tree.get_diameter(i)
        if d_p <= 0.0:
            continue
        d_a = tree.get_diameter(kids[0])
        d_b = tree.get_diameter(kids[1])
        rows.append((dist[i], (d_a * d_a + d_b * d_b) / (d_p * d_p)))
    return np.array(rows, dtype=[("length", "f8"), ("ratio", "f8")])

## Panel (a) — Mature tree at 400 yr

Grow the champion once for 400 generations and render the canopy
coloured by Strahler order via the existing `plot_tree_3d` helper.


In [ ]:
tree_mature = mt.grow_tree(
    cfg,
    n_generations=GEN_MATURE,
    seed=SEED,
    safety=safety,
    allocation=allocation,
)

print(
    f"{GEN_MATURE} yr tree: {tree_mature.get_number_of_branches()} branches, "
    f"{tree_mature.get_total_leaves()} leaves, "
    f"max Strahler = {mt.horton_summary(tree_mature).max_order}"
)
fig_a = mt.plot_tree_3d(tree_mature, style="cylinders")
fig_a.update_layout(title=f"{GEN_MATURE} yr champion (species_id={non_coding['species_id']})")
fig_a.show()

## Panels (b)–(e) — A second run with per-generation snapshots

We re-grow with the same seed but for 500 generations, snapshotting
`horton_summary` every 5 generations. The 25-yr snapshot is the open
markers in panel (b); the 400-yr `tree_mature` from above contributes
the filled markers, the tapering scatter in panel (d) and the
bifurcation table in panel (e).


In [ ]:
on_step, history = snapshot_cb(every=5)
tree_500 = mt.grow_tree(
    cfg,
    n_generations=GEN_MAX,
    seed=SEED,
    safety=safety,
    allocation=allocation,
    on_step=on_step,
)

# n_w  — classical Horton-Strahler stream count: n_w[0] == leaves,
#         n_w[w] = # internal branches with two children of equal Strahler
#         order w (Fortran NsegmentsS, mod_tree.f90:1052-1070).
# L_w  — mean stream length per Strahler rank in unit-twig units
#         (= n_segments_w / n_w), the Fortran length_Strahler
#         (mod_tree.f90:1091-1097). Drop-in for R_l in horton_ratios.
# horton_summary  — kept for the mean_diameter / mean_area series feeding
#         R_d / R_a fits.
n_w_400 = mt.horton_strahler_counts(tree_mature)
L_w_400 = mt.mean_stream_length(tree_mature)
summary_400 = mt.horton_summary(tree_mature)

# 25 yr snapshot from the 500-yr history (closest to GEN_YOUNG).
snap_25 = min(history, key=lambda h: abs(h["gen"] - GEN_YOUNG))
n_w_25 = snap_25["n_w"]
L_w_25 = snap_25["L_w"]
summary_25 = snap_25["summary"]

# Fits: n_w → R_n; L_w → R_l; summary mean_area/mean_diameter → R_a/R_d.
# max_rank=7 matches SI Fig. S12 (top ranks sit on few branches, noisy).
FIT_MAX_RANK = 7
fit_400 = mt.horton_ratios(
    summary_400,
    mean_length_override=L_w_400,
    n_branches_override=n_w_400,
    max_rank=FIT_MAX_RANK,
)
fit_25 = mt.horton_ratios(
    summary_25,
    mean_length_override=L_w_25,
    n_branches_override=n_w_25,
    max_rank=FIT_MAX_RANK,
)
print(
    f"tree_mature: {tree_mature.get_number_of_branches()} branches, "
    f"{tree_mature.get_total_leaves()} leaves, max Strahler={summary_400.max_order}"
)
print(f"  Horton-Strahler stream counts: {list(n_w_400)}  (rank 1 == leaves)")
print(f"  Mean stream length L(w):       {[round(float(x), 2) for x in L_w_400]}")
print()
print("Paper targets:  R_n=3.4  R_l=1.7  R_d=1.9  R_a=3.5  D=2.4")
print(
    f" 25 yr fit:    R_n={fit_25.R_n:.2f}  R_l={fit_25.R_l:.2f}  "
    f"R_d={fit_25.R_d:.2f}  R_a={fit_25.R_a:.2f}  D={fit_25.D:.2f}"
)
print(
    f"400 yr fit:    R_n={fit_400.R_n:.2f}  R_l={fit_400.R_l:.2f}  "
    f"R_d={fit_400.R_d:.2f}  R_a={fit_400.R_a:.2f}  D={fit_400.D:.2f}"
)

### Build the 2×2 panel

Plotly does not mix 3D and 2D subplots cleanly, so panel (a) lives in
its own figure above. Panels (b)–(e) share one `make_subplots` grid.


In [ ]:
fig = mt.figstyle.subplots(
    size="full",
    aspect=10 / 8,
    rows=2,
    cols=2,
    subplot_titles=[
        "(b) Architectural self-similarity",
        "(c) Evolution of fractal dimension",
        "(d) Branch tapering",
        "(e) Area conservation (Leonardo)",
    ],
    horizontal_spacing=0.12,
    vertical_spacing=0.14,
)


# ------------- Panel (b): per-order quantities + fits -------------------
# All four traces are indexed by Strahler / Horton rank w:
#   - # branches  → n_w  (Fortran NsegmentsS; rank-1 == leaves)
#   - <l>         → L_w  (Fortran length_Strahler; mean unit-twigs per stream)
#   - mean area, mean diameter → horton_summary fields (per-segment)
def _series_b(n_w, L_w, horton_s, name_suffix, symbol_open=False, showlegend=True):
    w_n = np.arange(1, len(n_w) + 1)
    w_L = np.arange(1, len(L_w) + 1)
    w_h = np.arange(1, horton_s.max_order + 1)
    marker_style = dict(size=9, line=dict(width=1.2))
    if symbol_open:
        # Open markers: only the outline, white fill.
        marker_style["color"] = "white"
    trace_specs = [
        ("# branches", w_n, n_w, "circle", mt.figstyle.COLORS["red"]),
        ("mean length", w_L, L_w, "diamond", mt.figstyle.COLORS["blue"]),
        ("mean area", w_h, horton_s.mean_area, "triangle-up", mt.figstyle.COLORS["red_25"]),
        (
            "mean diameter",
            w_h,
            horton_s.mean_diameter,
            "triangle-down",
            mt.figstyle.COLORS["blue_light"],
        ),
    ]
    for label, xs, ys, sym, line_color in trace_specs:
        marker = dict(marker_style)
        marker["symbol"] = sym
        marker["line"] = dict(width=1.4, color=line_color)
        if not symbol_open:
            marker["color"] = line_color
        fig.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="markers",
                name=f"{label} {name_suffix}",
                marker=marker,
                showlegend=showlegend,
                legendgroup=label,
            ),
            row=1,
            col=1,
        )


_series_b(n_w_25, L_w_25, summary_25, "(25 yr)", symbol_open=True, showlegend=True)
_series_b(n_w_400, L_w_400, summary_400, "(400 yr)", symbol_open=False, showlegend=True)

# Reference fit lines from the mature tree, drawn over fit_orders. Each
# line anchors at the rank-1 value of the series it represents and grows
# (or shrinks for # branches) by the fitted ratio per rank.
w_fit = fit_400.fit_orders
fit_lines = [
    (n_w_400, fit_400.R_n, -1, mt.figstyle.COLORS["red"]),
    (L_w_400, fit_400.R_l, +1, mt.figstyle.COLORS["blue"]),
    (summary_400.mean_area, fit_400.R_a, +1, mt.figstyle.COLORS["red_25"]),
    (summary_400.mean_diameter, fit_400.R_d, +1, mt.figstyle.COLORS["blue_light"]),
]
for arr, ratio, sign, color in fit_lines:
    anchor = float(arr[0])
    yline = anchor * ratio ** (sign * (w_fit - 1))
    fig.add_trace(
        go.Scatter(
            x=w_fit,
            y=yline,
            mode="lines",
            line=dict(color=color, width=1, dash="dash"),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )

fig.add_annotation(
    xref="x",
    yref="y",
    x=1.2,
    y=n_w_400[0] * 0.35,
    text=(
        f"R_n = {fit_400.R_n:.2f}<br>"
        f"R_l = {fit_400.R_l:.2f}<br>"
        f"R_d = {fit_400.R_d:.2f}<br>"
        f"R_a = {fit_400.R_a:.2f}<br>"
        f"D   = {fit_400.D:.2f}"
    ),
    showarrow=False,
    align="left",
    font=dict(size=10),
    bgcolor="rgba(255,255,255,0.7)",
    row=1,
    col=1,
)
fig.update_xaxes(title_text="Strahler / Horton rank w", row=1, col=1)
fig.update_yaxes(title_text="value", type="log", row=1, col=1)

# ------------- Panel (c): time evolution of R_n, R_l, D -----------------
t_arr, R_n_arr, R_l_arr, D_arr = [], [], [], []
R_n_band, R_l_band, D_band = [], [], []
for h in history:
    s = h["summary"]
    n_w = h["n_w"]
    L_w = h["L_w"]
    if s.max_order < 4:
        continue
    try:
        fit = mt.horton_ratios(
            s,
            mean_length_override=L_w,
            n_branches_override=n_w,
            max_rank=FIT_MAX_RANK,
        )
    except (ValueError, ZeroDivisionError):
        continue
    t_arr.append(h["gen"])
    R_n_arr.append(fit.R_n)
    R_l_arr.append(fit.R_l)
    D_arr.append(fit.D)
    last = min(s.max_order - 1, FIT_MAX_RANK)
    n_pairs = n_w[: last - 1] / np.where(n_w[1:last] > 0, n_w[1:last], 1)
    l_pairs = L_w[1:last] / np.where(L_w[: last - 1] > 0, L_w[: last - 1], 1)
    R_n_band.append(np.std(n_pairs[np.isfinite(n_pairs) & (n_pairs > 0)]))
    R_l_band.append(np.std(l_pairs[np.isfinite(l_pairs) & (l_pairs > 0)]))
    D_band.append(
        abs(fit.D)
        * np.sqrt(
            (R_n_band[-1] / max(fit.R_n, 1e-9) / np.log(max(fit.R_n, 1.001))) ** 2
            + (R_l_band[-1] / max(fit.R_l, 1e-9) / np.log(max(fit.R_l, 1.001))) ** 2
        )
    )

t_arr = np.asarray(t_arr)
for label, y, band, color in [
    ("R_n", R_n_arr, R_n_band, mt.figstyle.COLORS["red"]),
    ("R_l", R_l_arr, R_l_band, mt.figstyle.COLORS["blue"]),
    ("D", D_arr, D_band, mt.figstyle.COLORS["black"]),
]:
    y = np.asarray(y)
    band = np.asarray(band)
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([t_arr, t_arr[::-1]]),
            y=np.concatenate([y + band, (y - band)[::-1]]),
            fill="toself",
            fillcolor=color,
            opacity=0.12,
            line=dict(color="rgba(0,0,0,0)"),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=t_arr,
            y=y,
            mode="markers",
            name=label,
            line=dict(color=color, width=2),
            marker=dict(size=4),
        ),
        row=1,
        col=2,
    )

for gen, txt in [(GEN_YOUNG, "25 yr"), (GEN_MATURE, "400 yr")]:
    fig.add_vline(
        x=gen,
        line=dict(color=mt.figstyle.COLORS["grey"], width=1, dash="dot"),
        annotation_text=txt,
        annotation_position="top",
        row=1,
        col=2,
    )
fig.update_xaxes(title_text="time (yr)", row=1, col=2)
# Clamp the y-range. R_l → 1 sends D to infinity, so the early-time
# error bands blow up and plotly auto-scaling hides the steady-state values.
fig.update_yaxes(title_text="ratio", range=[0, 5], row=1, col=2)

# ------------- Panel (d): branch tapering -------------------------------
strahler, length, diameter = per_branch_arrays(tree_mature)
mask = (length > 0) & (diameter > 0)
strahler = strahler[mask]
length = length[mask]
diameter = diameter[mask]

for k in range(1, int(strahler.max()) + 1):
    sub = strahler == k
    if not sub.any():
        continue
    fig.add_trace(
        go.Scatter(
            x=diameter[sub],
            y=length[sub],
            mode="markers",
            marker=dict(
                size=4, color=mt.figstyle.strahler_color(k), opacity=0.6, line=dict(width=0)
            ),
            name=f"order {k}",
            legendgroup=f"d-order-{k}",
            showlegend=False,
        ),
        row=2,
        col=1,
    )

PANEL_D_SLOPES = [
    (2 / 3, "2/3"),
    # (1.0, "1"),
    # (3 / 2, "3/2"),
]
D_ANCHOR, L_ANCHOR = 10.0, 100.0
d_ref = np.array([0.01, 10.0])
for slope, label in PANEL_D_SLOPES:
    l_ref = L_ANCHOR * (d_ref / D_ANCHOR) ** slope
    fig.add_trace(
        go.Scatter(
            x=d_ref,
            y=l_ref,
            mode="lines",
            line=dict(color=mt.figstyle.COLORS["black"], width=1.2),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=2,
        col=1,
    )
    fig.add_annotation(
        x=d_ref[0],
        y=l_ref[0],
        text=f"slope {label}",
        showarrow=False,
        font=dict(size=9),
        xanchor="left",
        yanchor="bottom",
        row=2,
        col=1,
    )
fig.update_xaxes(
    title_text="diameter d",
    type="log",
    range=[-2, 1],
    dtick=1,
    exponentformat="power",
    showexponent="all",
    row=2,
    col=1,
)
fig.update_yaxes(
    title_text="⟨ℓ⟩",
    type="log",
    range=[0, 2],
    dtick=1,
    exponentformat="power",
    showexponent="all",
    row=2,
    col=1,
)

# ------------- Panel (e): area conservation -----------------------------
tbl = bifurcation_table(tree_mature)
PANEL_E_LMIN = 1.5
if tbl.size:
    big = tbl["length"] > PANEL_E_LMIN
    mean_ratio = float(tbl["ratio"][big].mean()) if big.any() else float("nan")
else:
    mean_ratio = float("nan")
fig.add_trace(
    go.Scatter(
        x=tbl["length"],
        y=tbl["ratio"],
        mode="markers",
        marker=dict(size=3, color=mt.figstyle.COLORS["blue"], opacity=0.35, line=dict(width=0)),
        showlegend=False,
        name="A_kids / A_parent",
    ),
    row=2,
    col=2,
)
fig.add_hline(
    y=mean_ratio,
    line=dict(color=mt.figstyle.COLORS["red"], width=1.8),
    annotation_text=f"mean (⟨ℓ⟩>{PANEL_E_LMIN}) = {mean_ratio:.2f}",
    annotation_position="bottom right",
    annotation_font=dict(color=mt.figstyle.COLORS["red"]),
    row=2,
    col=2,
)
fig.add_hline(
    y=1.0,
    line=dict(color=mt.figstyle.COLORS["grey"], width=1, dash="dot"),
    row=2,
    col=2,
)
fig.update_xaxes(
    title_text="parent ⟨ℓ⟩",
    type="log",
    range=[0, 2],
    dtick=1,
    exponentformat="power",
    showexponent="all",
    row=2,
    col=2,
)
fig.update_yaxes(
    title_text="area ratio",
    type="log",
    range=[-1, 1],
    dtick=1,
    exponentformat="power",
    showexponent="all",
    row=2,
    col=2,
)

fig.update_layout(
    # title=f"SI Fig. S8 — champion species_id={non_coding['species_id']}, seed={SEED}",
    legend=dict(font=dict(size=10)),
)
fig.show()

## Discussion

The 400-yr fitted ratios should sit within roughly ±20 % of the paper
targets `R_n = 3.5`, `R_l = 1.7`, `R_d = 1.9`, `R_a = 3.5`, `D = 2.3`
— close enough to confirm that the Python port recovers the published
self-similarity. Single-seed scatter is real; averaging across seeds
tightens the bands.

Panel (b) shows the qualitative point of the paper directly: the
*open* markers (25 yr, juvenile) lie off the geometric trend lines
while the *filled* markers (400 yr, mature) line up. Panel (c) traces
that same convergence over time — the ratios drift toward their
asymptotic values within ~200 generations.

Panel (e) typically lands near 0.95: cross-section is not perfectly
conserved at branchings in this model, but it is close. The horizontal
gray dotted line at 1.0 marks the strict Leonardo limit; the deficit
reflects the model's preference for slightly thinner daughters.


## Next steps

- Swap to `species_id=1` in cell 1 and re-run to compare the two S3
  champion clusters side-by-side.
- Re-run with the canopy-aware momentum wind by adding a
  ``wind: { model: momentum, ... }`` block to a forked
  `forest.yaml` — `grow_tree` picks it up automatically.
- Sweep `SEED` over 10–20 values and aggregate `fit_400` to get true
  population bands on each ratio.
- Compare against `notebooks/05_strahler_diagnostics.ipynb` for the
  per-Strahler-order (per-segment) view, which is appropriate for the
  Leonardo histogram but cannot resolve length ratios since every
  segment in MechaTree is a unit twig.
